# Herencia en Python (PCAP) — Teoría + ejemplos

Este cuaderno explica **herencia en Python** de forma clara y con ejemplos, siguiendo los puntos que suelen considerarse esenciales para la **certificación PCAP**.



## Contenidos <a id="contenidos"></a>

1. [¿Qué es la herencia y para qué se usa?](#que-es-la-herencia)
2. [Relación **is-a**: cuándo tiene sentido heredar](#relacion-is-a)
3. [Sintaxis de herencia (simple)](#sintaxis-de-herencia)
4. [Atributos y métodos heredados](#atributos-y-metodos-heredados)
5. [Sobrescritura (override) de métodos](#sobrescritura-de-metodos)
6. [`super()` y reutilización de comportamiento](#super-y-reutilizacion)
7. [Constructores en herencia: `__init__`](#constructores-en-herencia)
8. [Polimorfismo: misma interfaz, distinto comportamiento](#polimorfismo)
9. [Composición vs herencia (idea básica)](#composicion-vs-herencia)
10. [`isinstance()` e `issubclass()`](#isinstance-e-issubclass)
11. [Visibilidad y herencia: `_` y `__` (name mangling)](#visibilidad-y-herencia)
12. [Herencia múltiple](#herencia-multiple)
13. [MRO (Method Resolution Order) y cómo consultarlo](#mro)
14. [Todas las clases heredan de `object`](#todas-heredan-de-object)
15. [Métodos especiales y herencia (`__str__`, `__repr__`, etc.)](#metodos-especiales-y-herencia)

<a id="que-es-la-herencia"></a>
## 1) ¿Qué es la herencia y para qué se usa?

La **herencia** permite crear una clase nueva (**subclase**) a partir de otra existente (**superclase** o **clase base**).

- La subclase **hereda** atributos y métodos de la clase base.
- La subclase puede **añadir** comportamiento (nuevos métodos/atributos).
- La subclase puede **modificar** comportamiento heredado (sobrescritura / override).

### Ventajas típicas
- Reutilización de código.
- Jerarquías de clases más claras.
- Posibilidad de polimorfismo (ver sección 8).

### Riesgos típicos
- Jerarquías demasiado profundas o mal diseñadas.
- Usar herencia cuando lo correcto sería composición (ver sección 9).

[Ir al índice](#contenidos)

In [ ]:
# Ejemplo mínimo: una clase base y una subclase

class Animal:
    def respirar(self):
        return "Respirando..."

class Perro(Animal):
    pass

p = Perro()
print(p.respirar())  # Heredado de Animal


<a id="relacion-is-a"></a>
## 2) Relación **is-a**: cuándo tiene sentido heredar

Una regla práctica: la herencia suele tener sentido cuando se cumple la relación **"es un/a" (is-a)**.

- `Perro` **es un** `Animal` → ✅ herencia plausible.
- `Motor` **es un** `Coche` → ❌ normalmente no; un coche *tiene un* motor (composición).

> En Python, aunque técnicamente se puede heredar “de casi cualquier cosa”, conviene que la jerarquía tenga sentido conceptual.

[Ir al índice](#contenidos)

In [ ]:
# Ejemplo de 'is-a'
class Vehiculo:
    def arrancar(self):
        return "Arrancando..."

class Coche(Vehiculo):
    def tocar_claxon(self):
        return "Piiii!"

c = Coche()
print(c.arrancar())      # Vehiculo
print(c.tocar_claxon())  # Coche



<a id="sintaxis-de-herencia"></a>
## 3) Sintaxis de herencia (simple)

La herencia simple se define colocando la clase base entre paréntesis:

```python
class Subclase(ClaseBase):
    ...
```

- La subclase hereda métodos y atributos.
- Si la subclase define un método con el mismo nombre que la base, lo **sobrescribe** (override).

[Ir al índice](#contenidos)


In [ ]:
class Base:
    def saludo(self):
        return "Hola desde Base"

class Derivada(Base):
    pass

d = Derivada()
print(d.saludo())


<a id="atributos-y-metodos-heredados"></a>
## 4) Atributos y métodos heredados

La subclase recibe:
- **métodos** definidos en la base,
- **atributos de clase** definidos en la base,
- (y, si se inicializan) **atributos de instancia** creados por el `__init__` de la base (si se ejecuta).

### Atributos de clase vs atributos de instancia
- **Atributo de clase**: se define dentro de la clase, fuera de métodos.
- **Atributo de instancia**: se asigna normalmente dentro de `__init__` usando `self`.

[Ir al índice](#contenidos)

In [ ]:
class Persona:
    especie = "Homo sapiens"  # atributo de clase

    def __init__(self, nombre):
        self.nombre = nombre  # atributo de instancia

    def presentarse(self):
        return f"Soy {self.nombre}."

class Estudiante(Persona):
    pass

e = Estudiante("Marina")
print(e.especie)         # heredado (atributo de clase)
print(e.nombre)          # creado por __init__ de Persona (atributo de instancia)
print(e.presentarse())   # heredado (método)


## 5) Sobrescritura (override) de métodos <a id="sobrescritura-de-metodos"></a>


Si una subclase define un método con el mismo nombre que uno de la base:
- el método de la subclase **reemplaza** al de la base para instancias de la subclase.

Esto se usa para:
- especializar comportamiento,
- adaptar el resultado,
- imponer reglas adicionales.

[Ir al índice](#contenidos)

In [ ]:
class Mensajero:
    def enviar(self, mensaje):
        return f"Enviando: {mensaje}"

class MensajeroSeguro(Mensajero):
    def enviar(self, mensaje):
        # Comportamiento distinto en la subclase
        return f"Enviando cifrado: ***{len(mensaje)} caracteres***"

m1 = Mensajero()
m2 = MensajeroSeguro()

print(m1.enviar("Hola"))
print(m2.enviar("Hola"))


## 6) `super()` y reutilización de comportamiento <a id="super-y-reutilizacion"></a>

`super()` permite acceder a métodos de la clase base **sin nombrarla directamente**.

Se usa mucho para:
- reutilizar parte de un método sobrescrito,
- encadenar inicializaciones en `__init__`,
- trabajar correctamente en herencia múltiple (ver sección 12 y 13).

> Idea: en vez de copiar y pegar lógica del padre, se llama al método del padre y se amplía.

[Ir al índice](#contenidos)

In [ ]:
class Logger:
    def log(self, mensaje):
        return f"[LOG] {mensaje}"

class LoggerConNivel(Logger):
    def log(self, mensaje):
        base = super().log(mensaje)  # llama al método de la clase base
        return base + " (nivel=INFO)"

l = LoggerConNivel()
print(l.log("Sistema iniciado"))


## 7) Constructores en herencia: `__init__` <a id="constructores-en-herencia"></a>

`__init__` es el método de inicialización de la instancia.

Casos típicos:
1. **La subclase no define `__init__`** → se usa el `__init__` de la base.
2. **La subclase define `__init__`** → si se quiere inicializar también lo de la base, se llama a `super().__init__(...)`.

> Si una subclase sobrescribe `__init__` y no llama a `super().__init__`, los atributos que la base inicializa **no existirán**, a menos que se creen manualmente.

[Ir al índice](#contenidos)


In [ ]:
class Cuenta:
    def __init__(self, titular, saldo=0):
        self.titular = titular
        self.saldo = saldo

    def ingresar(self, cantidad):
        self.saldo += cantidad
        return self.saldo

class CuentaAhorro(Cuenta):
    def __init__(self, titular, saldo=0, interes=0.02):
        super().__init__(titular, saldo)  # inicializa la parte heredada
        self.interes = interes            # añade un atributo propio

    def aplicar_interes(self):
        self.saldo *= (1 + self.interes)
        return self.saldo

ca = CuentaAhorro("José", 100, interes=0.05)
print(ca.titular, ca.saldo, ca.interes)
print(ca.aplicar_interes())


In [ ]:
# Ejemplo de problema común: sobrescribir __init__ sin llamar a super()

class BaseInit:
    def __init__(self):
        self.x = 10

class MalaSubclase(BaseInit):
    def __init__(self):
        # No se llama a super().__init__()
        self.y = 99

m = MalaSubclase()
print("¿Tiene y?", hasattr(m, "y"))
print("¿Tiene x?", hasattr(m, "x"))  # False: no se inicializó en la base


## 8) Polimorfismo: misma interfaz, distinto comportamiento <a id="polimorfismo"></a>

El **polimorfismo** consiste en poder usar objetos de distintas clases mediante la **misma interfaz** (mismos nombres de métodos), obteniendo comportamientos distintos según la clase real del objeto.

En Python esto encaja muy bien con el enfoque “duck typing”:
> *Si se comporta como un pato (tiene el método esperado), se puede usar como un pato.*

En PCAP suele ser importante entender:
- cómo la herencia facilita polimorfismo,
- cómo el override activa el comportamiento específico.

[Ir al índice](#contenidos)

In [ ]:
class Notificador:
    def enviar(self, texto):
        raise NotImplementedError("Debe implementarse en subclases")

class Email(Notificador):
    def enviar(self, texto):
        return f"Email enviado: {texto}"

class SMS(Notificador):
    def enviar(self, texto):
        return f"SMS enviado: {texto}"

def avisar(notificador, texto):
    # No importa la clase exacta: importa que tenga .enviar()
    print(notificador.enviar(texto))

avisar(Email(), "Reunión a las 10:00")
avisar(SMS(), "Código: 1234")


## 9) Composición vs herencia (idea básica) <a id="composicion-vs-herencia"></a>

- **Herencia**: una clase **es un** tipo de otra (is-a).
- **Composición**: una clase **tiene un** objeto de otra.

La composición se usa cuando:
- se quiere reutilizar funcionalidad sin formar una jerarquía,
- se busca menor acoplamiento,
- se quiere cambiar fácilmente componentes internos.

En muchos diseños, **composición** es más flexible que herencia.

[Ir al índice](#contenidos)

In [ ]:
# Ejemplo: composición (Coche tiene un Motor)

class Motor:
    def encender(self):
        return "Motor encendido"

class CocheConMotor:
    def __init__(self, motor):
        self.motor = motor  # composición

    def arrancar(self):
        return self.motor.encender() + " -> Coche listo"

c = CocheConMotor(Motor())
print(c.arrancar())


## 10) `isinstance()` e `issubclass()` <a id="isinstance-e-issubclass"></a>

- `isinstance(obj, Clase)` comprueba si `obj` es instancia de `Clase` **o de alguna subclase**.
- `issubclass(Sub, Super)` comprueba si `Sub` hereda (directa o indirectamente) de `Super`.

Son útiles para:
- validaciones,
- depuración,
- lógica condicional basada en jerarquías.


In [ ]:
class A: ...
class B(A): ...

b = B()

print(isinstance(b, B))  # True
print(isinstance(b, A))  # True (B hereda de A)
print(issubclass(B, A))  # True
print(issubclass(A, B))  # False


## 11) Visibilidad y herencia: `_` y `__` (name mangling) <a id="visibilidad-y-herencia"></a>

En Python no existe “private” como en algunos lenguajes, pero hay convenciones:

### `_atributo` (convención de “protegido”)
- Significa: “uso interno” o “no tocar desde fuera”.
- **Se puede** acceder, pero se considera mala práctica hacerlo desde fuera si no es necesario.
- En herencia, una subclase puede usar `_atributo` del padre.

### `__atributo` (doble guion bajo): *name mangling*
- Python cambia el nombre internamente para evitar colisiones en subclases.
- `__x` dentro de `Clase` se transforma en `_Clase__x`.

Esto no lo hace totalmente privado, pero:
- reduce colisiones de nombres,
- dificulta el acceso accidental.

[Ir al índice](#contenidos)

In [ ]:
class Padre:
    def __init__(self):
        self._protegido = "Se puede usar en subclases (por convención)"
        self.__privado = "Name mangling activo"

    def ver_privado(self):
        return self.__privado

class Hija(Padre):
    def mostrar(self):
        parte1 = self._protegido

        existe_directo = hasattr(self, "__privado")
        existe_mangled = hasattr(self, "_Padre__privado")
        valor_mangled = self._Padre__privado

        return parte1, existe_directo, existe_mangled, valor_mangled

h = Hija()
print(h.mostrar())
print(h.ver_privado())   # forma recomendada: método del padre


## 12) Herencia múltiple <a id="herencia-multiple"></a>

Python permite heredar de varias clases:

```python
class C(A, B):
    ...
```

Esto es potente, pero puede complicar:
- el orden en el que se buscan métodos,
- las inicializaciones (`__init__`),
- la mantenibilidad.

Por eso se recomienda usarlo con criterio, normalmente para “mezclar” comportamientos (mixins).

[Ir al índice](#contenidos)

In [ ]:
class Nadador:
    def mover(self):
        return "Nadando"

class Corredor:
    def mover(self):
        return "Corriendo"

class Triatleta(Nadador, Corredor):
    pass

t = Triatleta()
print(t.mover())  # ¿Cuál se elige? Depende del MRO (ver sección 13)
print(t.mover(Corredor)) # MRO explícito: elige el método de Corredor

## 13) MRO (Method Resolution Order) y cómo consultarlo <a id="mro"></a>

El **MRO** es el orden que sigue Python para buscar un atributo o método cuando se accede a él.

- En herencia simple, suele ser “subclase → base → object”.
- En herencia múltiple, el orden se calcula con el algoritmo C3 (linealización), para mantener consistencia.

Se puede consultar con:
- `Clase.mro()`
- `Clase.__mro__`

[Ir al índice](#contenidos)

In [1]:
class A1:
    def quien(self):
        return "A1"

class B1(A1):
    def quien(self):
        return "B1"

class C1(A1):
    def quien(self):
        return "C1"

class D1(B1, C1):
    pass

d = D1()
print(d.quien())      # se elige según el MRO
print(D1.mro())       # lista de clases en el orden de búsqueda
print(D1.__mro__)     # tupla equivalente


B1
[<class '__main__.D1'>, <class '__main__.B1'>, <class '__main__.C1'>, <class '__main__.A1'>, <class 'object'>]
(<class '__main__.D1'>, <class '__main__.B1'>, <class '__main__.C1'>, <class '__main__.A1'>, <class 'object'>)


In [ ]:
# MRO del ejemplo de Triatleta (sección 12)

class Nadador2:
    def mover(self):
        return "Nadando"

class Corredor2:
    def mover(self):
        return "Corriendo"

class Triatleta2(Nadador2, Corredor2):
    pass

print(Triatleta2.mro())
print(Triatleta2().mover())  # primero Nadador2 porque aparece antes en la herencia


## 14) Todas las clases heredan de `object` <a id="todas-heredan-de-object"></a>

En Python 3, **todas** las clases derivan (directa o indirectamente) de `object`.

Esto implica que se heredan comportamientos básicos como:
- representación,
- comparación por identidad,
- etc.

Se puede comprobar con `isinstance(obj, object)` o mirando el MRO.

[Ir al índice](#contenidos)


In [ ]:
class MiClase:
    pass

x = MiClase()
print(isinstance(x, object))  # True
print(MiClase.mro())          # normalmente: [MiClase, object]


## 15) Métodos especiales y herencia (`__str__`, `__repr__`, etc.) <a id="metodos-especiales-y-herencia"></a>

Los **métodos especiales** (dunder methods) pueden heredarse y sobrescribirse.

Dos muy comunes:
- `__str__`: representación “amigable” para usuarios (por ejemplo al hacer `print(obj)`).
- `__repr__`: representación más técnica, útil para depuración.

Sobrescribirlos mejora la legibilidad y el diagnóstico.

[Ir al índice](#contenidos)

In [ ]:
class Producto:
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio

    def __str__(self):
        return f"{self.nombre} - {self.precio:.2f} €"

    def __repr__(self):
        return f"Producto(nombre={self.nombre!r}, precio={self.precio!r})"

class ProductoOferta(Producto):
    def __init__(self, nombre, precio, descuento):
        super().__init__(nombre, precio)
        self.descuento = descuento

    def __str__(self):
        base = super().__str__() #Sobrescribiendo el metodo del padre para aumentarlo. 
        return base + f" (descuento {self.descuento*100:.0f}%)"

p = Producto("Camiseta", 19.99)
o = ProductoOferta("Camiseta", 19.99, 0.20)

print(str(p))
print(repr(p))
print(str(o))
print(repr(o))  # heredado de Producto (no se sobrescribió __repr__ en la subclase)


Camiseta - 19.99 €
Producto(nombre='Camiseta', precio=19.99)
Camiseta - 19.99 € (descuento 20%)
Producto(nombre='Camiseta', precio=19.99)


NameError: name 'Object' is not defined

## Resumen de ideas clave (para repasar)

- Herencia: subclase hereda y puede extender/sobrescribir.
- `super()` para reutilizar lógica del padre y encadenar inicializaciones.
- `__init__`: si se sobrescribe, normalmente se llama a `super().__init__`.
- Polimorfismo: misma interfaz, distinto comportamiento.
- Composición vs herencia: “tiene un” vs “es un”.
- `isinstance` / `issubclass` para comprobar jerarquías.
- `_` (convención) y `__` (name mangling) en contexto de herencia.
- Herencia múltiple requiere entender MRO.
- Todo hereda de `object`.
- Métodos especiales se heredan y pueden sobrescribirse.

[Ir al índice](#contenidos)